# Victim Agent 3: OpenHands

OpenHands is a **full victim agent app/runtime**: web UI, backend, Docker sandbox/runtime, tools, and model configuration. It is not an attack agent.

Unlike AgentDojo and mini-SWE-agent, OpenHands is heavier. It normally requires Docker Desktop or `uv tool install openhands`. This notebook checks prerequisites and gives explicit launch commands.

In [ ]:

from __future__ import annotations
import json, os, subprocess, sys
from pathlib import Path

REPO = Path.cwd()
if not (REPO / "pyproject.toml").exists():
    REPO = REPO.parent
assert (REPO / "pyproject.toml").exists(), REPO
VENV_PY = REPO / ".venv" / "bin" / "python"
print("repo:", REPO)
print("venv python:", VENV_PY, "exists=", VENV_PY.exists())

def run(cmd, timeout=120, env=None):
    merged = os.environ.copy()
    if env:
        merged.update(env)
    p = subprocess.run(cmd, cwd=REPO, text=True, capture_output=True, timeout=timeout, env=merged)
    print("$", " ".join(map(str, cmd)))
    print("returncode:", p.returncode)
    if p.stdout:
        print("STDOUT:\n", p.stdout[-4000:])
    if p.stderr:
        print("STDERR:\n", p.stderr[-4000:])
    return p

def load_raw_key(path=Path("api_keys")):
    path = REPO / path
    if not path.exists():
        return None
    for line in path.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if line and not line.startswith("#"):
            if "=" in line:
                return line.split("=", 1)[1].strip().strip("'\"")
            return line
    return None


## 1. Prerequisite Check

In [ ]:
run(["which", "docker"], timeout=20)
run(["which", "uv"], timeout=20)
run(["which", "openhands"], timeout=20)

## 2. How To Install / Launch

Recommended official path:

```bash
uv tool install openhands --python 3.12
openhands serve --mount-cwd
```

Alternative Docker path requires Docker Desktop and pulls large images. After launch, the local GUI is usually at `http://localhost:3000`. Configure Qwen in the UI using:

- provider/model: OpenAI-compatible / custom model
- model: `qwen3.7-plus`
- base URL: `https://dashscope-intl.aliyuncs.com/compatible-mode/v1`
- API key: from `api_keys`

## 3. Optional Launch Cell

This is disabled by default because it can download large Docker images and starts a long-running server.

In [ ]:
RUN_OPENHANDS = False
if RUN_OPENHANDS:
    run(["openhands", "serve", "--mount-cwd"], timeout=20)
else:
    print("Skipped. Install Docker/uv/OpenHands first, then set RUN_OPENHANDS=True or run the command in a terminal.")

## Status

- Installed now: depends on prerequisite check. On the current machine, earlier checks showed `docker` and `uv` were not found.
- Victim status: real victim agent app/runtime once installed.
- Notebook role: prerequisite checker and launch guide; not yet a fully automated smoke harness.